**Prophet**:
- Prophet is a procedure for forecasting time series data based on an additive model where non-linear trends are fit with yearly, weekly, and daily seasonality, plus holiday effects. It works best with time series that have strong seasonal effects and several seasons of historical data. Prophet is robust to missing data and shifts in the trend, and typically handles outliers well.
- Maybe don't need one that handles missing data if we don't have any

# Inputs

In [ ]:
# Minimum viable dataframe -> two mandatory columns minimum

#Our data should already contain the datetime as index and all features as columns

#Target as extra input also with datetime index

import pandas as pd

df = pd.DataFrame({
    'ds': pd.date_range('2019-01-01', periods=8760, freq='h'),
    'y':  wholesale_price_array   # your target, numeric
})

# ds must be a proper datetime, not a string
df['ds'] = pd.to_datetime(df['ds'])

df.head()

# Setup

In [ ]:
# Install if needed: pip install prophet
from prophet import Prophet

m = Prophet(
    seasonality_mode='multiplicative',   # or 'additive', but for volatile peaks multiplicative should work best
    daily_seasonality=False,         # we'll add custom below
    weekly_seasonality=True,
    yearly_seasonality=True,
)

m.fit(df)   # takes 1–10 sec on 5 years of hourly data

# Hourly and Seasonality

In [ ]:
m = Prophet(
    daily_seasonality=True,   # disable built-in
    weekly_seasonality=True,
    yearly_seasonality=True,
)

# Add hourly cycle (24h period)
m.add_seasonality(
    name='hourly',
    period=1,           # 1 day = 24 periods for hourly data
    fourier_order=8,    # start at 8, tune later. Fourier order = how many sine/cosine waves to use.
)

# Also consider a weekly-hour interaction
m.add_seasonality(
    name='weekly_hourly',
    period=7,
    fourier_order=5,
)

#regressors tell Prophet which columns are used as feature. this loop adds all columns as feature
exclude = ['ds', 'y']  # CHANGE TO ACTUAL NAMES OF DATETIME AND WHOLESALEPIRCE-TARGET COLUMN
regressors = [col for col in df.columns if col not in exclude]

for col in regressors:
    m.add_regressor(col)

# Fit

In [ ]:
m.fit(df)

# Corss validate

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

df_cv = cross_validation(
    m,
    initial='1800 days',   # train on first 2 years
    period='90 days',     # retrain every 30 days
    horizon='1 days',     # evaluate up to 24h ahead
    parallel='processes'  # speeds it up significantly
)

#get and print performance matrix
metrics = performance_metrics(df_cv)
print(metrics[['horizon', 'mae', 'rmse', 'mape']])

In [ ]:
#plot error

from prophet.plot import plot_cross_validation_metric
plot_cross_validation_metric(df_cv, metric='mae')

# Predict

In [ ]:
# We might split it beforehand so we have 24 hours of feature to use to predict the price

# Build a future dataframe for next 24 hours
future = m.make_future_dataframe(
    periods=24,
    freq='h',
    include_history=False   # just the forecast window
)

# Attach regressor forecasts for those 24 hours
future['demand_north']      = demand_forecast_24h
future['wind_generation']    = wind_forecast_24h
future['hydro_storage']     = hydro_now   # relatively static
future['temperature_auckland'] = temp_forecast_24h

forecast = m.predict(future)

# Key output columns:
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]